# End-to-End Training

This notebook trains a compact classifier on the bundled Wine dataset. It follows the same API as Hello World, but uses a few more numeric inputs and exposes root embeddings during the same run.

Import the runtime pieces used in the full training loop: Lightning for optimization, Polars for tabular records, and scikit-learn for a local dataset.

In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint
from sklearn.datasets import load_wine

import json2vec as j2v

logger.remove()

The Wine dataset is still flat, but it gives enough numeric variation to make a clearer supervised example than handmade records. The model will use four chemical measurements to predict `cultivar`.

In [2]:
ROOT = "[*][*]"
wine = load_wine()
indices = []
for target_id in range(len(wine.target_names)):
    indices.extend(index for index, target in enumerate(wine.target) if target == target_id)
indices = indices[:48]

records = pl.DataFrame(
    {
        "alcohol": wine.data[indices, 0].tolist(),
        "malic_acid": wine.data[indices, 1].tolist(),
        "color_intensity": wine.data[indices, 9].tolist(),
        "proline": wine.data[indices, 12].tolist(),
        "cultivar": [wine.target_names[index] for index in wine.target[indices]],
    }
)

records.head()

alcohol,malic_acid,color_intensity,proline,cultivar
f64,f64,f64,f64,str
14.23,1.71,5.64,1065.0,"""class_0"""
13.2,1.78,4.38,1050.0,"""class_0"""
13.16,2.36,5.68,1185.0,"""class_0"""
14.37,1.95,7.8,1480.0,"""class_0"""
13.24,2.59,4.32,735.0,"""class_0"""


The schema is the architecture. Four `Number` requests feed the root encoder, `cultivar` is the categorical target, and `model.set(..., embed=True)` asks the root node to return embeddings after training.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("alcohol", query=f"{ROOT}.alcohol"),
    j2v.Number("malic_acid", query=f"{ROOT}.malic_acid"),
    j2v.Number("color_intensity", query=f"{ROOT}.color_intensity"),
    j2v.Number("proline", query=f"{ROOT}.proline"),
    j2v.Category("cultivar", query=f"{ROOT}.cultivar", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)
_ = model.set(j2v.where("name") == "record", embed=True)

`PolarsDataModule.from_model(...)` reads the schema configuration from the model, so batch size, queries, targets, and tensorfield behavior stay tied to one object.

In [4]:
datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

The tutorial trains for one tiny pass. In a real experiment this is where you would scale epochs, validation splits, callbacks, logging, and checkpointing.

In [5]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

The plot confirms that the target and root embedding are both configured before inference.

In [6]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  26,039                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  5                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               

After training, `predict` returns typed outputs for supervised targets and `embed` returns vectors from the nodes configured with `embed=True`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))
pprint(model.embed(batch))

{
│   'record/cultivar': {
│   │   'state': {
│   │   │   'valued': [0.028422944247722626, 0.028422944247722626, 0.028422951698303223],
│   │   │   'null': [0.07059528678655624, 0.07059528678655624, 0.07059528678655624],
│   │   │   'padded': [0.39351165294647217, 0.39351165294647217, 0.39351165294647217],
│   │   │   'masked': [0.11809071153402328, 0.11809071153402328, 0.11809071153402328],
│   │   │   'other': [0.3893794119358063, 0.3893794119358063, 0.3893793523311615]
│   │   },
│   │   'content': {'value': [None, None, None], 'probability': [0.0, 0.0, 0.0], 'topk': [[], [], []]}
│   }
}

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.053043536841869354,
│   │   │   │   0.12936250865459442,
│   │   │   │   0.302306592464447,
│   │   │   │   0.3460499346256256,
│   │   │   │   -0.38336578011512756,
│   │   │   │   0.15812404453754425,
│   │   │   │   0.232753723859787,
│   │   │   │   0.16601966321468353,
│   │   │   │   -0.33291444182395935,
│   │   │   │   0.048104915767908096,
│   │   │   │   0.19000987708568573,
│   │   │   │   0.08008299767971039,
│   │   │   │   -0.22183287143707275,
│   │   │   │   -0.5225436687469482,
│   │   │   │   -0.0869988352060318,
│   │   │   │   -0.17351937294006348
│   │   │   ],
│   │   │   [
│   │   │   │   0.053043536841869354,
│   │   │   │   0.12936250865459442,
│   │   │   │   0.302306592464447,
│   │   │   │   0.3460499346256256,
│   │   │   │   -0.38336578011512756,
│   │   │   │   0.15812404453754425,
│   │   │   │   0.232753723859787,
│   │   │   │   0.16601966321468353,
│   │   │   │   -0.33291444182395935,
│   │   │   │   0.048104915767908096,
│   │   │   │   0.19000987708568573,
│   │   │   │   0.08008299767971039,
│   │   │   │   -0.22183287143707275,
│   │   │   │   -0.5225436687469482,
│   │   │   │   -0.0869988352060318,
│   │   │   │   -0.17351937294006348
│   │   │   ],
│   │   │   [
│   │   │   │   0.05304352939128876,
│   │   │   │   0.12936247885227203,
│   │   │   │   0.30230656266212463,
│   │   │   │   0.3460499048233032,
│   │   │   │   -0.3833657205104828,
│   │   │   │   0.15812402963638306,
│   │   │   │   0.23275363445281982,
│   │   │   │   0.16601961851119995,
│   │   │   │   -0.33291444182395935,
│   │   │   │   0.04810488224029541,
│   │   │   │   0.19000989198684692,
│   │   │   │   0.08008305728435516,
│   │   │   │   -0.22183288633823395,
│   │   │   │   -0.5225436091423035,
│   │   │   │   -0.08699885755777359,
│   │   │   │   -0.1735193431377411
│   │   │   ]
│   │   ]
│   }
}